### Get training samples from Guacamol

In [25]:
import pandas as pd
import numpy as np
from rdkit import Chem
import safe
from utils import canonicalize_smiles, convert_smiles_to_safe
random_state = 42

In [ ]:
#---Read Guacamol train set and sample 2000 SMILES after canonicalization
df = pd.read_csv("data/input/guacamol_v1_train.smiles", header=None, names=["smiles"])
df = df.sample(n=2000, random_state=random_state).reset_index(drop=True)
df['canonical_smiles'] = df['smiles'].apply(canonicalize_smiles)
df['safe'] = df['canonical_smiles'].apply(convert_smiles_to_safe)
df = df.drop(columns=["smiles"])
print("Rows:", len(df))
print("Unique SAFE:", df.nunique().safe)
df.head(5)

Rows: 2000
Unique SAFE: 1948


,canonical_smiles,safe
0,O[Se]c1ccccc1,None
1,ClCc1ccccc1Cn1cnc2c(Cl)ncnc21,n13cnc2c(Cl)ncnc21.c14ccccc15.ClC4.C35
2,CC1(c2ccccc2F)COC(N)=N1,CC13COC(N)=N1.c13ccccc1F
3,COC(=O)CCNC(=O)c1nnn(-c2ccccc2)c1C(F)(F)F,c16nnn8c17.c18ccccc1.C3(=O)CC5.C7(F)(F)F.C46=O...
4,C=C1C(=O)OC2C1CCC(C)C13OC(c4ccc(Cl)cc4)OC1C(=C...,C=C1C(=O)OC2C1CCC(C)C13OC6OC1C=5C(=O)C23C.C=5c...


In [27]:
#---Drop non-unique and null SAFE values
df = df.dropna(subset=['safe']).drop_duplicates(subset=['safe'])
df.head(5)

,canonical_smiles,safe
1,ClCc1ccccc1Cn1cnc2c(Cl)ncnc21,n13cnc2c(Cl)ncnc21.c14ccccc15.ClC4.C35
2,CC1(c2ccccc2F)COC(N)=N1,CC13COC(N)=N1.c13ccccc1F
3,COC(=O)CCNC(=O)c1nnn(-c2ccccc2)c1C(F)(F)F,c16nnn8c17.c18ccccc1.C3(=O)CC5.C7(F)(F)F.C46=O...
4,C=C1C(=O)OC2C1CCC(C)C13OC(c4ccc(Cl)cc4)OC1C(=C...,C=C1C(=O)OC2C1CCC(C)C13OC6OC1C=5C(=O)C23C.C=5c...
5,CCN(CCCN)CCCNc1nccc2c(C)c3[nH]c4ccc(OC)cc4c3cc12,c1%10nccc2c(C)c3[nH]c4ccc5cc4c3cc12.C7CCN.C8CC...


In [28]:
#---sample data splits

train_1000 = df.sample(n=1000, random_state=random_state)
remaining_df = df.drop(train_1000.index)

valid_200 = remaining_df.sample(n=200, random_state=random_state)
remaining_df = remaining_df.drop(valid_200.index)

test_200 = remaining_df.sample(n=200, random_state=random_state)

In [29]:
#---confirm that no sets have any overlapping SMILES
s_train = set(train_1000['canonical_smiles'])
s_valid = set(valid_200['canonical_smiles'])
s_test = set(test_200['canonical_smiles'])

no_common_samples = s_train.isdisjoint(s_valid) and s_train.isdisjoint(s_test) and s_valid.isdisjoint(s_test)
print("No common SMILES between train, valid and test sets:", no_common_samples)

#---confirm that no sets have any overlapping SAFE
s_train = set(train_1000['safe'])
s_valid = set(valid_200['safe'])
s_test = set(test_200['safe'])

no_common_samples = s_train.isdisjoint(s_valid) and s_train.isdisjoint(s_test) and s_valid.isdisjoint(s_test)
print("No common SAFE between train, valid and test sets:", no_common_samples)

No common SMILES between train, valid and test sets: True
No common SAFE between train, valid and test sets: True


In [ ]:
train_1000.to_csv("data/input/part1_arm_a_train_1K.csv", index=False, header=True, encoding="utf8", columns=["canonical_smiles", "safe"])
valid_200.to_csv("data/input/valid_200.csv", index=False, header=True, encoding="utf8", columns=["canonical_smiles", "safe"])
test_200.to_csv("data/input/test_200.csv", index=False, header=True, encoding="utf8", columns=["canonical_smiles", "safe"])